# Tugas Besar 1 — IF3270 Pembelajaran Mesin
## Pengujian Feedforward Neural Network (FFNN) From Scratch

## 1. Setup & Imports

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import random
from nn import FFNN

random.seed(42)
np.random.seed(42)

## 2. Preprocessing Dataset

### 2.1 Load Dataset

In [ ]:
df = pd.read_csv('../data/datasetml_2026.csv')
print(f'Shape: {df.shape}')
df.head()

### 2.2 Explorasi Data

In [ ]:
print(df.info())
print('\nValue Counts Target')
print(df['placement_status'].value_counts())
print('\nMissing Values')
print(df.isnull().sum())

### 2.3 Encoding & Feature Engineering

In [ ]:
# One-hot
df['placement_status'] = df['placement_status'].map({'Placed': 1, 'Not Placed': 0})

categorical_cols = ['college_tier', 'country', 'university_ranking_band', 'specialization', 'industry']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(f'Shape after encoding: {df_encoded.shape}')
df_encoded.head()

### 2.4 Split & Normalize

In [ ]:
X = df_encoded.drop('placement_status', axis=1).values
y = df_encoded['placement_status'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split: 80:20 train val
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train_list = X_train.tolist()
X_val_list = X_val.tolist()
y_train_list = [[yi] for yi in y_train.tolist()]
y_val_list = [[yi] for yi in y_val.tolist()]

n_features = X_train.shape[1]
print(f'Features: {n_features}')
print(f'Train: {len(X_train_list)}, Val: {len(X_val_list)}')
print(f'Class distribution (train): {sum(y_train)}/{len(y_train)} placed')

## 3. Helper Functions

In [ ]:
def plot_loss_history(history, title='Training & Validation Loss'):
    plt.figure(figsize=(10, 5))
    plt.plot(history['train_loss'], label='Train Loss', linewidth=2)
    if history['val_loss'][0] is not None:
        plt.plot(history['val_loss'], label='Val Loss', linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def evaluate_model(model, X, y_true, label=''):
    preds = model.predict(X)
    y_pred = [1 if p[0] > 0.5 else 0 for p in preds]
    acc = accuracy_score(y_true, y_pred)
    print(f'{label} Accuracy: {acc:.4f}')
    return acc, y_pred

def train_and_evaluate(model, X_train, y_train, X_val, y_val, epochs=10, lr=0.01, batch_size=64, loss_fn='bce', optimizer='sgd', verbose=1, reg_type=None, reg_lambda=0.0, title='Model'):
    history = model.fit(X_train, y_train, epochs=epochs, learning_rate=lr, batch_size=batch_size, loss_fn=loss_fn, optimizer=optimizer, X_val=X_val, y_val=y_val, verbose=verbose, reg_type=reg_type, reg_lambda=reg_lambda)
    plot_loss_history(history, title=f'{title} - Loss')
    train_acc, _ = evaluate_model(model, X_train, [y[0] for y in y_train], label=f'{title} Train')
    val_acc, _ = evaluate_model(model, X_val, [y[0] for y in y_val], label=f'{title} Val')
    return history, train_acc, val_acc

## 4. Eksperimen 1: Pengaruh Depth & Width

### 4.1 Variasi Width (Depth Tetap = 3 hidden layers)

In [ ]:
# 3 variasi width, depth tetap
EPOCHS = 100
LR = 0.01
BATCH_SIZE = 32
width_configs = {'Width 8':  [n_features, 8, 8, 8, 1], 'Width 16': [n_features, 16, 16, 16, 1], 'Width 32': [n_features, 32, 32, 32, 1],}
width_results = {}
for name, sizes in width_configs.items():
    print(f'\n{name}')
    activations = ['relu'] * (len(sizes) - 2) + ['sigmoid']
    model = FFNN(sizes, activations, init_method='uniform',
                 init_params={'low': -0.5, 'high': 0.5}, seed=42)
    hist, train_acc, val_acc = train_and_evaluate(
        model, X_train_list, y_train_list, X_val_list, y_val_list,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, loss_fn='bce',
        verbose=1, title=name)
    width_results[name] = {'history': hist, 'train_acc': train_acc, 'val_acc': val_acc, 'model': model}

#### Perbandingan Loss per Epoch (Width)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, res in width_results.items():
    axes[0].plot(res['history']['train_loss'], label=name)
    axes[1].plot(res['history']['val_loss'], label=name)
axes[0].set_title('Train Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title('Val Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('Pengaruh Width pada Loss')
plt.tight_layout()
plt.show()

print('\nRangkuman Akurasi')
for name, res in width_results.items():
    print(f'{name}: Train={res["train_acc"]:.4f}, Val={res["val_acc"]:.4f}')

### 4.2 Variasi Depth (Width Tetap = 16)

In [ ]:
# 3 variasi depth, width tetap
depth_configs = {'Depth 1': [n_features, 16, 1], 'Depth 3': [n_features, 16, 16, 16, 1], 'Depth 5': [n_features, 16, 16, 16, 16, 16, 1],}

depth_results = {}
for name, sizes in depth_configs.items():
    print(f'\n{name}')
    activations = ['relu'] * (len(sizes) - 2) + ['sigmoid']
    model = FFNN(sizes, activations, init_method='uniform',
                 init_params={'low': -0.5, 'high': 0.5}, seed=42)
    hist, train_acc, val_acc = train_and_evaluate(
        model, X_train_list, y_train_list, X_val_list, y_val_list,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, loss_fn='bce',
        verbose=1, title=name)
    depth_results[name] = {'history': hist, 'train_acc': train_acc, 'val_acc': val_acc, 'model': model}

#### Perbandingan Loss per Epoch (Depth)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, res in depth_results.items():
    axes[0].plot(res['history']['train_loss'], label=name)
    axes[1].plot(res['history']['val_loss'], label=name)
axes[0].set_title('Train Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title('Val Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('Pengaruh Depth pada Loss')
plt.tight_layout()
plt.show()

print('\nRangkuman Akurasi')
for name, res in depth_results.items():
    print(f'{name}: Train={res["train_acc"]:.4f}, Val={res["val_acc"]:.4f}')

## 5. Eksperimen 2: Pengaruh Fungsi Aktivasi Hidden Layer

Base arsitektur: 3 hidden layers [n_features, 16, 16, 16, 1]. Variasi aktivasi pada layer ke-2 (index 1).

In [ ]:
activation_names = ['linear', 'relu', 'sigmoid', 'tanh']
activation_results = {}

for act_name in activation_names:
    print(f'\nActivation: {act_name}')
    activations = ['relu', act_name, 'relu', 'sigmoid']
    sizes = [n_features, 16, 16, 16, 1]
    model = FFNN(sizes, activations, init_method='uniform',
                 init_params={'low': -0.5, 'high': 0.5}, seed=42)
    hist, train_acc, val_acc = train_and_evaluate(
        model, X_train_list, y_train_list, X_val_list, y_val_list,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, loss_fn='bce',
        verbose=1, title=f'Activation={act_name}')
    activation_results[act_name] = {'history': hist, 'train_acc': train_acc, 'val_acc': val_acc}

#### Perbandingan Loss per Epoch (Aktivasi)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, res in activation_results.items():
    axes[0].plot(res['history']['train_loss'], label=name)
    axes[1].plot(res['history']['val_loss'], label=name)
axes[0].set_title('Train Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title('Val Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('Pengaruh Fungsi Aktivasi pada Loss')
plt.tight_layout()
plt.show()

print('\nRangkuman Akurasi')
for name, res in activation_results.items():
    print(f'{name}: Train={res["train_acc"]:.4f}, Val={res["val_acc"]:.4f}')

## 6. Eksperimen 3: Pengaruh Learning Rate

In [ ]:
learning_rates = [0.001, 0.01, 0.1]
lr_results = {}

for lr in learning_rates:
    sizes = [n_features, 16, 16, 16, 1]
    activations = ['relu', 'relu', 'relu', 'sigmoid']
    model = FFNN(sizes, activations, init_method='uniform',
                 init_params={'low': -0.5, 'high': 0.5}, seed=42)
    hist, train_acc, val_acc = train_and_evaluate(
        model, X_train_list, y_train_list, X_val_list, y_val_list,
        epochs=EPOCHS, lr=lr, batch_size=BATCH_SIZE, loss_fn='bce',
        verbose=1, title=f'LR={lr}')
    lr_results[lr] = {'history': hist, 'train_acc': train_acc, 'val_acc': val_acc, 'model': model}

#### Perbandingan Loss per Epoch (Learning Rate)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for lr, res in lr_results.items():
    axes[0].plot(res['history']['train_loss'], label=f'LR={lr}')
    axes[1].plot(res['history']['val_loss'], label=f'LR={lr}')
axes[0].set_title('Train Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title('Val Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('Pengaruh Learning Rate pada Loss')
plt.tight_layout()
plt.show()

print('\nRangkuman Akurasi')
for lr, res in lr_results.items():
    print(f'LR={lr}: Train={res["train_acc"]:.4f}, Val={res["val_acc"]:.4f}')

#### Distribusi Bobot & Gradien (Learning Rate)

In [ ]:
for lr, res in lr_results.items():
    print(f'\nLR={lr}')
    model = res['model']
    model.plot_weight_distribution()
    model.plot_gradient_distribution()

## 7. Eksperimen 4: Pengaruh Regularisasi

In [ ]:
reg_configs = {'No Reg': {'reg_type': None, 'reg_lambda': 0.0}, 'L1 (lambda=0.001)': {'reg_type': 'l1', 'reg_lambda': 0.001}, 'L2 (lambda=0.001)': {'reg_type': 'l2', 'reg_lambda': 0.001},}

reg_results = {}
for name, config in reg_configs.items():
    print(f'\n{name}')
    sizes = [n_features, 16, 16, 16, 1]
    activations = ['relu', 'relu', 'relu', 'sigmoid']
    model = FFNN(sizes, activations, init_method='uniform',
                 init_params={'low': -0.5, 'high': 0.5}, seed=42)
    hist, train_acc, val_acc = train_and_evaluate(
        model, X_train_list, y_train_list, X_val_list, y_val_list,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, loss_fn='bce',
        verbose=1, title=name,
        reg_type=config['reg_type'], reg_lambda=config['reg_lambda'])
    reg_results[name] = {'history': hist, 'train_acc': train_acc, 'val_acc': val_acc, 'model': model}

#### Perbandingan Loss per Epoch (Regularisasi)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, res in reg_results.items():
    axes[0].plot(res['history']['train_loss'], label=name)
    axes[1].plot(res['history']['val_loss'], label=name)
axes[0].set_title('Train Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title('Val Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('Pengaruh Regularisasi pada Loss')
plt.tight_layout()
plt.show()

for name, res in reg_results.items():
    print(f'{name}: Train={res["train_acc"]:.4f}, Val={res["val_acc"]:.4f}')

#### Distribusi Bobot & Gradien (Regularisasi)

In [ ]:
for name, res in reg_results.items():
    print(name)
    model = res['model']
    model.plot_weight_distribution()
    model.plot_gradient_distribution()

## 8. Eksperimen 5: Pengaruh Optimizer (Adam vs SGD)


In [ ]:
opt_configs = {'SGD': 'sgd','Adam': 'adam'}

opt_results = {}
for name, opt in opt_configs.items():
    print(f'\nOptimizer: {name}')
    sizes = [n_features, 16, 16, 16, 1]
    activations = ['relu', 'relu', 'relu', 'sigmoid']
    model = FFNN(sizes, activations, init_method='uniform',
                 init_params={'low': -0.5, 'high': 0.5}, seed=42)
    hist, train_acc, val_acc = train_and_evaluate(
        model, X_train_list, y_train_list, X_val_list, y_val_list,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, loss_fn='bce',
        verbose=1, title=name, optimizer=opt)
    opt_results[name] = {'history': hist, 'train_acc': train_acc, 'val_acc': val_acc, 'model': model}


#### Perbandingan Loss per Epoch (Optimizer)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, res in opt_results.items():
    axes[0].plot(res['history']['train_loss'], label=name)
    axes[1].plot(res['history']['val_loss'], label=name)
axes[0].set_title('Train Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title('Val Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('Pengaruh Optimizer pada Loss (Adam vs SGD)')
plt.tight_layout()
plt.show()

print('\nRangkuman Akurasi')
for name, res in opt_results.items():
    print(f'{name}: Train={res["train_acc"]:.4f}, Val={res["val_acc"]:.4f}')


## 9. Uji Perbandingan: FFNN vs sklearn MLPClassifier

In [ ]:
from sklearn.neural_network import MLPClassifier

sizes = [n_features, 16, 16, 16, 1]
activations = ['relu', 'relu', 'relu', 'sigmoid']
our_model = FFNN(sizes, activations, init_method='uniform',
                 init_params={'low': -0.5, 'high': 0.5}, seed=42)
our_hist = our_model.fit(X_train_list, y_train_list, epochs=EPOCHS,
                         learning_rate=LR, batch_size=BATCH_SIZE,
                         loss_fn='bce', X_val=X_val_list, y_val=y_val_list, verbose=1)
our_train_acc, _ = evaluate_model(our_model, X_train_list, y_train.tolist(), 'Our FFNN Train')
our_val_acc, our_pred = evaluate_model(our_model, X_val_list, y_val.tolist(), 'Our FFNN Val')

sklearn_model = MLPClassifier(
    hidden_layer_sizes=(16, 16, 16),
    activation='relu',
    solver='sgd',
    learning_rate_init=LR,
    batch_size=BATCH_SIZE,
    max_iter=EPOCHS,
    random_state=42,
    verbose=False
)
sklearn_model.fit(X_train, y_train)
sklearn_train_acc = sklearn_model.score(X_train, y_train)
sklearn_val_acc = sklearn_model.score(X_val, y_val)
sklearn_pred = sklearn_model.predict(X_val)
print(f'sklearn Train Accuracy: {sklearn_train_acc:.4f}')
print(f'sklearn Val Accuracy: {sklearn_val_acc:.4f}')

### Perbandingan Hasil

In [ ]:
print(f'Our FFNN: Train={our_train_acc:.4f}, Val={our_val_acc:.4f}')
print(f'sklearn MLP: Train={sklearn_train_acc:.4f}, Val={sklearn_val_acc:.4f}')

print('\nClassification Report (Our FFNN)')
print(classification_report(y_val, our_pred, target_names=['Not Placed', 'Placed']))

print('Classification Report (sklearn MLP)')
print(classification_report(y_val, sklearn_pred, target_names=['Not Placed', 'Placed']))

## 10. Eksperimen 6: Pengaruh RMSNorm


In [ ]:
norm_configs = {
    'Tanpa RMSNorm': False,
    'Dengan RMSNorm': True
}

norm_results = {}
for name, use_rmsnorm in norm_configs.items():
    print(f'\n{name}')
    sizes = [n_features, 16, 16, 16, 1]
    activations = ['relu', 'relu', 'relu', 'sigmoid']
    model = FFNN(sizes, activations, init_method='uniform',
                 init_params={'low': -0.5, 'high': 0.5}, seed=42, use_rmsnorm=use_rmsnorm)
    hist, train_acc, val_acc = train_and_evaluate(
        model, X_train_list, y_train_list, X_val_list, y_val_list,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, loss_fn='bce',
        verbose=1, title=name)
    norm_results[name] = {'history': hist, 'train_acc': train_acc, 'val_acc': val_acc, 'model': model}


#### Analisis: Loss, Akurasi, dan Distribusi Gradien


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, res in norm_results.items():
    axes[0].plot(res['history']['train_loss'], label=name)
    axes[1].plot(res['history']['val_loss'], label=name)
axes[0].set_title('Train Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title('Val Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('Pengaruh RMSNorm pada Loss')
plt.tight_layout()
plt.show()

print('\nRangkuman Akurasi Akhir')
for name, res in norm_results.items():
    print(f'{name}: Train={res["train_acc"]:.4f}, Val={res["val_acc"]:.4f}')

for name, res in norm_results.items():
    print(name)
    res['model'].plot_weight_distribution()
    res['model'].plot_gradient_distribution()
